# Exploring Features Before Modeling

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import mutual_info_regression
from gridcast.config import get_connection
from gridcast.features import features as ft
import plotly.express as px

In [ ]:
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT * FROM analytics.features;")
columns = [desc[0] for desc in cur.description]
featureset = pd.DataFrame(cur.fetchall(), columns=columns)

features = ft(featureset, drop_time=False)

In [ ]:
trainrange = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D').tz_localize(None)
valrange = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D').tz_localize(None)
testrange = pd.date_range(start='2026-01-01', end='2026-08-10', freq='D').tz_localize(None)

df = features['time'].dt.tz_localize(None).dt.normalize()

train = features[df.isin(trainrange)]
validation = features[df.isin(valrange)]
test = features[df.isin(testrange)]

In [ ]:
train.isna().sum()

In [ ]:
train = train[~train.demand_lag_168h.isna()]

In [ ]:
train.drop(columns=['time', 'observation_count'], inplace=True)

In [ ]:
y = train['demand_mw']
X = train.drop(['demand_mw'], axis=1)

In [ ]:
X

In [ ]:
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = X[col].astype('category').cat.codes

discrete = X.dtypes == 'category'

# mi = mutual_info_regression(X, y, discrete_features=discrete, random_state=42)
mi

In [ ]:
results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

In [ ]:
pd.DataFrame(results)

In [ ]:
train.dtypes

In [ ]:
discrete = train.dtypes != 'float64'

In [ ]:
px.bar(x = results.index, y=results.values, title='Mutual Info – Feature Importance')

In [ ]:
lis = discrete[discrete == True]

In [ ]:
lis

In [ ]:
train.drop(columns=list(lis.index)).corr()